In [1]:
!cp "/content/drive/MyDrive/Hate Speech LLMs/Data ICWSM/Eval Data/gptData.xlsx" .

In [2]:
!pip install hazm bert_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.9 MB/s eta 0:00:00
  Created wheel for flashtext: filename=flashtext-2.7-py2.py3-none-any.whl size=9300 sha256=0f1e68340fdf029fd5f15077ba2e8f00e03b9c479d5c3af60b2f17799753f25e
  Stored in directory: /root/.cache/pip/wheels/be/12/c9/228313ff5cb722777830302f1d4136de4129932e6a0354c056
Successfully built flashtext


In [3]:
!pip install -q "transformers<5" "tokenizers<0.22" "bert-score==0.3.12"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import numpy as np
import pandas as pd
import ast
from hazm import WordTokenizer
from bert_score import score as bert_score
import torch

In [5]:
df = pd.read_excel("gptData.xlsx")

In [6]:
ref_col = 'Hop_5_Rationale_Moral_Span (Ref)'
pred_col = 'Hop_5_Rationale_Moral_Span'

In [7]:
df = df[df["Hop_1_Triage (Ref)"] != "بی‌ضرر"]
df = df[df["Hop_1_Triage"] != "بی‌ضرر"]
print(len(df))

882


In [8]:
df[ref_col] = df[ref_col].apply(ast.literal_eval)
df[pred_col] = df[pred_col].apply(ast.literal_eval)

In [9]:
tokenizer = WordTokenizer()

def tokenize(text):
    return tokenizer.tokenize(text)

In [10]:
def jaccard_similarity(a, b):
    A = set(tokenize(a))
    B = set(tokenize(b))
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

def max_pairwise_jaccard(pred_spans, gold_spans):
    if not pred_spans or not gold_spans:
        return 0.0

    return max(
        jaccard_similarity(p, g)
        for p in pred_spans
        for g in gold_spans
    )

In [11]:
def max_pairwise_bertscore(
    pred_spans,
    gold_spans,
    model_type="HooshvareLab/bert-base-parsbert-uncased",
    device=None
):
    if not pred_spans or not gold_spans:
        return 0.0

    max_score = 0.0

    for p in pred_spans:
        P, R, F1 = bert_score(
            [p] * len(gold_spans),
            gold_spans,
            model_type=model_type,
            num_layers=12,
            lang='fa',
            verbose=False,
            device=device,
        )
        max_score = max(max_score, F1.max().item())

    return max_score

In [12]:
def evaluate_rationales(df, gold_col, pred_col):
    jaccards = []
    bertscores = []

    for index, row in df.iterrows():
        print(index)
        max_pairwise_jaccard_val = max_pairwise_jaccard(row[pred_col], row[gold_col])
        jaccards.append(max_pairwise_jaccard_val)

        bertScore = max_pairwise_bertscore(row[pred_col], row[gold_col])
        bertscores.append(bertScore)

    return {
        "MaxPairwiseJaccard": sum(jaccards) / len(jaccards),
        "MaxPairwiseBERTScore": sum(bertscores) / len(bertscores)
    }

In [ ]:
results = evaluate_rationales(df, gold_col=ref_col, pred_col=pred_col)

In [14]:
results

{'MaxPairwiseJaccard': 0.6895858081002754,
 'MaxPairwiseBERTScore': 0.8062482209973324}

In [15]:
def jaccard(a, b):
    A, B = set(tokenize(a)), set(tokenize(b))
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

In [16]:
def normal_jaccard(pred_spans, gold_spans):
    if not pred_spans or not gold_spans:
        return 0.0

    scores = []
    for p in pred_spans:
        scores.append(max(jaccard(p, g) for g in gold_spans))
    return sum(scores) / len(scores)

In [17]:
def normal_bertscore(pred_spans, gold_spans, device=None):
    if not pred_spans or not gold_spans:
        return 0.0

    scores = []
    for p in pred_spans:
        _, _, F1 = bert_score(
            [p] * len(gold_spans),
            gold_spans,
            model_type="HooshvareLab/bert-base-parsbert-uncased",
            num_layers=12,
            lang='fa',
            verbose=False,
            device=device,
        )
        scores.append(F1.max().item())
    return sum(scores) / len(scores)

In [18]:
def evaluate_rationales2(df, gold_col, pred_col):
    jaccards = []
    bertscores = []

    for index, row in df.iterrows():
        print(index)
        jaccard_val = normal_jaccard(row[pred_col], row[gold_col])
        jaccards.append(jaccard_val)

        bertScore = normal_bertscore(row[pred_col], row[gold_col])
        bertscores.append(bertScore)

    return {
        "normalJaccard": sum(jaccards) / len(jaccards),
        "normalBERTScore": sum(bertscores) / len(bertscores)
    }

In [ ]:
results2 = evaluate_rationales2(df, gold_col=ref_col, pred_col=pred_col)

In [20]:
results2

{'normalJaccard': 0.5345225377850445, 'normalBERTScore': 0.7261823114725737}